# Add final labels and metadata onto raw PBMC object
### By [Aishwarya Chander](aishwarya.chander@alleninstitute.org), High Resolution Translational Immunology, Allen Institute for Immunology
**Main aim**: Merges the curated L1–L3 cell type labels from FH1 and BRI cohorts onto the raw PBMC AnnData, incorporates corrected monocyte/DC labels, adds CMV status, computes subject age, maps visit details to abbreviated timepoint labels (treatment and flu), handles the FH1011/FH1003 90-day timepoint edge cases, drops Other_DC and erythrocytes, and saves the final analysis-ready PBMC object. Also saves a 500k-cell subsample for visualization.

In [1]:
import pandas as pd
import hisepy as hp
import scanpy as sc
import numpy as np
sc.settings.n_jobs = 60

path = '../../../data/rna/'

In [2]:
file = ['54656ef9-46cc-4ec8-a7f1-5e58ec127bbb'] ## BRI Metadata UUID
pq_path = hp.cache_files(file)
bri_meta = pd.read_parquet(pq_path)

In [3]:
pbmc_meta = pd.read_parquet(path+'metadata/all-pbmc-raw-metadata.parquet')

ndmm_meta = pd.read_parquet(path+'pbmc-labels/mansi_singh_ndmm_pbmc.parquet')
fixed_mono_dc = pd.read_parquet(path+'pbmc-labels/mono_dc_labels.parquet')

In [4]:
ndmm_meta.loc[ndmm_meta.index.intersection(fixed_mono_dc.index), "tidy.aifi_l3"] = (
    fixed_mono_dc.loc[ndmm_meta.index.intersection(fixed_mono_dc.index), "tidy.aifi_l3"]
)

In [5]:
# Prepare individual label DataFrames
bri_pbmc_labels = bri_meta[['barcodes', 'AIFI_L3']].rename(
    columns={'AIFI_L3': 'tidy.aifi_l3'})
fh1_pbmc_labels = ndmm_meta[['tidy.aifi_l3']].reset_index().rename(columns={
    'index': 'barcodes'})
combined_labels = pd.concat([bri_pbmc_labels, fh1_pbmc_labels], axis=0)

# Filter to only barcodes in pbmc_meta
combined_labels = combined_labels[combined_labels['barcodes'].isin(
    pbmc_meta.index)]

### Label Mapping

In [7]:
# Build mapping and apply to pbmc_meta
label_map = dict(
    zip(combined_labels['barcodes'], combined_labels['tidy.aifi_l3']))
pbmc_meta['tidy.aifi_l3'] = pbmc_meta.index.map(label_map)
pbmc_meta = pbmc_meta.dropna(subset=['tidy.aifi_l3']).rename(
    columns={'tidy.aifi_l3': 'aifi_label_l3'})

In [8]:
tidy_aifi = pd.read_csv("aifi-tidy-nomenclature.csv")
label_names = tidy_aifi[["l1_label", "l2_label", "l3_label"]].drop_duplicates()
label_names = label_names.apply(
    lambda col: col.str.rstrip() if col.dtype == "object" else col
)

l3_to_l2 = dict(zip(label_names["l3_label"], label_names["l2_label"]))
pbmc_meta["aifi_label_l2"] = [
    l3_to_l2[x] if x in l3_to_l2 else x for x in pbmc_meta["aifi_label_l3"]
]

l2_to_l1 = dict(zip(label_names["l2_label"], label_names["l1_label"]))
pbmc_meta["aifi_label_l1"] = [
    l2_to_l1[x] if x in l2_to_l1 else x for x in pbmc_meta["aifi_label_l2"]
]

pbmc_meta["aifi_label_l1"] = pbmc_meta["aifi_label_l1"].replace(
    {
        "Core naive CD8 T cell": "T cell",
        "CD8aa": "T cell",
        "Other_DC": "DC",
        "Platelet": "Other",
        "Progenitor cell": "Other",
        "ILC": "Other",
        "Erythrocyte": "Other"
    }
)

In [9]:
comp_names = tidy_aifi[["l1_name", "l2_name", "l3_name"]].drop_duplicates()
l3_to_comp3 = dict(zip(label_names["l3_label"], comp_names["l3_name"]))
pbmc_meta["aifi_celltype_l3"] = [
    l3_to_comp3[x] if x in l3_to_comp3 else x for x in pbmc_meta["aifi_label_l3"]
]

l2_to_comp2 = dict(zip(label_names["l2_label"], comp_names["l2_name"]))
pbmc_meta["aifi_celltype_l2"] = [
    l2_to_comp2[x] if x in l2_to_comp2 else x for x in pbmc_meta["aifi_label_l2"]
]

l1_to_comp1 = dict(zip(label_names["l1_label"], comp_names["l1_name"]))
pbmc_meta["aifi_celltype_l1"] = [
    l1_to_comp1[x] if x in l1_to_comp1 else x for x in pbmc_meta["aifi_label_l1"]
]

In [10]:
pbmc_meta["aifi_celltype_l1"] = pbmc_meta["aifi_celltype_l1"].replace(
    {"Other": "other"}
)

pbmc_meta["aifi_celltype_l2"] = pbmc_meta["aifi_celltype_l2"].replace(
    {"CD8aa": "t_cd8_aa", "Other_DC": "dc_other"}
)


pbmc_meta["aifi_celltype_l3"] = pbmc_meta["aifi_celltype_l3"].replace(
    {"CD8aa": "t_cd8_aa", "Other_DC": "dc_other"}
)

In [11]:
l2_to_plot2 = {
    # T cell
    "CD8aa": "CD8aa",
    "DN T cell": "DN T",
    "MAIT": "MAIT",
    "Memory CD4 T cell": "CD4 T Memory",
    "Memory CD8 T cell": "CD8 T Memory",
    "Naive CD4 T cell": "CD4 T Naive",
    "Naive CD8 T cell": "CD8 T Naive",
    "Proliferating T cell": "Prolif T",
    "Treg": "Treg",
    "gdT": "gdT",
    # B cell
    "Effector B cell": "Effector B",
    "Memory B cell": "Memory B",
    "Naive B cell": "Naive B",
    "Plasma cell": "Plasma",
    "Transitional B cell": "Trans B",
    # Monocyte
    "CD14 monocyte": "CD14 Mono",
    "CD16 monocyte": "CD16 Mono",
    "Intermediate monocyte": "Int Mono",
    # NK cell
    "CD56bright NK cell": "CD56br NK",
    "CD56dim NK cell": "CD56dim NK",
    "Proliferating NK cell": "Prolif NK",
    # Dendritic cell
    "ASDC": "ASDC",
    "cDC1": "cDC1",
    "cDC2": "cDC2",
    "pDC": "pDC",
    "Other_DC": "Other_DC",
    # Other
    "Erythrocyte": "Erythrocyte",
    "ILC": "ILC",
    "Platelet": "Platelet",
    "Progenitor cell": "Progenitor",
}

pbmc_meta["aifi_plot_l2"] = (
    pbmc_meta["aifi_label_l2"]
    .map(l2_to_plot2)
    .astype("category")
    .cat.remove_unused_categories()
)

### Build out the object

In [12]:
adata = sc.read_h5ad(path + "raw-files/all-pbmc-raw.h5ad")

In [14]:
pbmc_cell_labels = pbmc_meta[
    [
        "aifi_label_l1",
        "aifi_celltype_l1",
        "aifi_label_l2",
        "aifi_celltype_l2",
        "aifi_plot_l2",
        "aifi_label_l3",
        "aifi_celltype_l3",
    ]
]

adata.obs = adata.obs.merge(pbmc_cell_labels, on="barcodes", how="left")
adata = adata[adata.obs["aifi_label_l1"].notna()]

In [15]:
# CMV serostatus per subject (source: clinical metadata)
cmv_df = pd.read_csv("../../../data/rna/metadata/subject_cmv_status.csv")
cmv_dict = dict(zip(cmv_df["subject_id"], cmv_df["cmv_status"]))
adata.obs["subject.cmv"] = adata.obs["subject.subjectGuid"].map(cmv_dict)

/tmp/ipykernel_194314/519988704.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs["subject.cmv"] = adata.obs["subject.subjectGuid"].map(cmv_dict)


In [16]:
visit_labels = {
    "MM Pre-Treatment": "PreTx",
    "MM Post Induction 2-Cycles": "PI2C",
    "MM End Induction 1st Draw": "EI",
    "MM Post Transplant 60 Days": "ASCT60d",
    "MM Post Transplant 90 Days": "ASCT90d",
    "MM Post Transplant 1 year": "ASCT1y",
    "MM Post Transplant 2 year": "ASCT2y",
    "Healthy": "Healthy",
}

adata.obs["label.visitDetails"] = (
    adata.obs["sample.visitDetails"]
    .map(visit_labels)
    .astype("category")
    .cat.remove_unused_categories()
)

In [17]:
# adata.obs[['aifi_celltype_l1', 'aifi_celltype_l2', 'aifi_celltype_l3']].drop_duplicates().reset_index().drop(['barcodes'], axis=1).to_csv('pbmc_mm_labels.csv')

In [18]:
pbmc_br_healthy_kits = [
    'KT00015', 'KT00025', 'KT00339', 'KT00352', 'KT00356', 'KT00368', 'KT00369', 'KT00383', 'KT00388', 'KT00390',
    'KT00516', 'KT00520', 'KT00526', 'KT00533', 'KT00537', 'KT00539', 'KT00541', 'KT00545', 'KT00557', 'KT00563',
    'KT00564', 'KT00566', 'KT00582', 'KT00588', 'KT00599', 'KT00600', 'KT00602', 'KT00625', 'KT00334', 'KT00353',
    'KT00501', 'KT00569'
]

adata.obs['label.visitDetails'] = adata.obs['label.visitDetails'].cat.add_categories('Healthy')
adata.obs.loc[adata.obs['sample.sampleKitGuid'].isin(pbmc_br_healthy_kits), 'label.visitDetails'] = 'Healthy'

In [19]:
adata.obs.loc[
        adata.obs['subject.subjectGuid'].isin(['FH1011', 'FH1003']),
        ['subject.subjectGuid', 'sample.visitDetails']
    ].drop_duplicates().sort_values(['subject.subjectGuid', 'sample.visitDetails'], na_position='last').reset_index(drop=True)

,subject.subjectGuid,sample.visitDetails
0,FH1003,MM End Induction 1st Draw
1,FH1003,MM Post Induction 2-Cycles
2,FH1003,MM Post Transplant 1 year
3,FH1003,MM Post Transplant 60 Days
4,FH1003,MM Post Transplant 90 Days
5,FH1003,MM Pre-Treatment
6,FH1003,N/A - Flu-Series Timepoint Only
7,FH1011,MM End Induction 1st Draw
8,FH1011,MM Post Induction 2-Cycles
9,FH1011,MM Post Transplant 1 year


> For FH11 and FH03, we have 90 day time points post transplant. All other PBMC samples only have 60d as their transplant cycle. For FH1011, we'll rename the 90 day time point to 60 days. Additionally, we'll drop the FH1003 90d timepoint and just use the 60 day visit.

In [20]:
adata.obs.loc[
    (adata.obs['subject.subjectGuid'] == 'FH1011') & 
    (adata.obs['label.visitDetails'] == 'ASCT90d'),
    'label.visitDetails'
] = 'ASCT60d'

In [21]:
mask = ~(
    (adata.obs['subject.subjectGuid'] == 'FH1003') &
    (adata.obs['label.visitDetails'] == 'ASCT90d')
)

adata = adata[mask]

In [22]:
adata.obs['sample.drawDate'] = pd.to_datetime(adata.obs['sample.drawDate'])
adata.obs['subject.birthYear'] = adata.obs['subject.birthYear'].astype(int)

# compute age
adata.obs['subject.age'] = (
    adata.obs['sample.drawDate'].dt.year - adata.obs['subject.birthYear']
)
adata.obs['sample.drawDate'] = adata.obs['sample.drawDate'].astype(str)

/tmp/ipykernel_194314/2952287252.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs['sample.drawDate'] = pd.to_datetime(adata.obs['sample.drawDate'])


In [23]:
adata.obs['sample.visitName'].value_counts()

Other - Non-Flu                      1127541
Flu Year 1 Day 90                     726002
Flu Year 1 Day 7                      702860
Flu Year 1 Day 0                      685776
Flu Year 2 Day 0                      632838
Flu Year 2 Day 7                      596633
Flu Year 2 Day 90                     563025
Flu Year 1 Stand-Alone                261306
Flu Year 2 Stand-Alone                162379
COVID-19 Visit 15 Post-transplant      18019
COVID-19 Visit 5 Pre-transplant        14774
COVID-19 Visit 4 Pre-transplant        14197
COVID-19 Visit 6 Pre-transplant        14174
COVID-19 Visit 7 Pre-transplant        14079
Name: sample.visitName, dtype: int64

### Short Hand Visit Labels

In [24]:
visit_name_sh = {
    "Flu Year 1 Day 0": "Flu_Y1D0",
    "Flu Year 1 Day 7": "Flu_Y1D7",
    "Other - Non-Flu": np.nan,
    "Flu Year 1 Day 90": "Flu_Y1D90",
    "Flu Year 1 Stand-Alone": "Flu_Y1SA",
    "Flu Year 2 Stand-Alone": "Flu_Y2SA",
    "Flu Year 2 Day 0": "Flu_Y2D0",
    "Flu Year 2 Day 7": "Flu_Y2D7",
    "Flu Year 2 Day 90": "Flu_Y2D90",
    "COVID-19 Visit 5 Pre-transplant": np.nan,
    "COVID-19 Visit 4 Pre-transplant": np.nan,
    "COVID-19 Visit 6 Pre-transplant": np.nan,
    "COVID-19 Visit 7 Pre-transplant": np.nan,
    "COVID-19 Visit 15 Post-transplant": np.nan,
}

adata.obs["label.visitName"] = adata.obs["sample.visitName"].map(visit_name_sh)

### Drop unnecessary populations

In [26]:
adata = adata[~adata.obs['aifi_plot_l2'].isin(['Other_DC', 'Erythrocyte'])]

In [27]:
# List of columns
cat_columns = [
    'aifi_label_l1', 
    'aifi_celltype_l1', 
    'aifi_label_l2', 
    'aifi_celltype_l2', 
    'aifi_plot_l2', 
    'aifi_label_l3', 
    'aifi_celltype_l3'
]

# Drop unused categories
for col in cat_columns:
    if col in adata.obs.columns:
        if adata.obs[col].dtype.name != 'category':
            adata.obs[col] = adata.obs[col].astype('category')
        adata.obs[col] = adata.obs[col].cat.remove_unused_categories()

/tmp/ipykernel_194314/1494916627.py:16: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata.obs[col] = adata.obs[col].astype('category')


In [28]:
adata.write(path+'final-objects/final-pbmc-raw.h5ad')
adata.obs.to_parquet(path+'final-objects/final-pbmc-metadata.parquet')

In [29]:
sc.pp.subsample(adata, n_obs=500000, copy=False)
adata.write(path+'pbmc-subsets/all-pbmc-filtered-500k.h5ad')